In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
torch.manual_seed(0)
np.random.seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

# Load and preprocess the dataset
data = load_breast_cancer()
X, y = data.data, data.target


cpu


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, 
    test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)
y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1).to(device)
y_test_t = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1).to(device)

print(f'X_train shape: {X_train_t.shape}, y_train shape: {y_train_t.shape}')
print(f'X_test shape: {X_test_t.shape}, y_test shape: {y_test_t.shape}')

X_train shape: torch.Size([455, 30]), y_train shape: torch.Size([455, 1])
X_test shape: torch.Size([114, 30]), y_test shape: torch.Size([114, 1])


In [6]:
%pip install torchinfo

In [7]:
# %pip install torchinfo
class LogisticRegression(nn.Module):
    def __init__(self, input_dim):
        super(LogisticRegression, self).__init__()
        self.linear = nn.Linear(input_dim, 1)

    def forward(self, x):
        return torch.sigmoid(self.linear(x))
    
input_dim = X_train_t.shape[1]
model = LogisticRegression(input_dim).to(device)

from torchinfo import summary
summary(model, input_size=(1, input_dim))

Layer (type:depth-idx)                   Output Shape              Param #
LogisticRegression                       [1, 1]                    --
├─Linear: 1-1                            [1, 1]                    31
Total params: 31
Trainable params: 31
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.00
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00

In [8]:
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

num_epochs = 200
train_losses = []
for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()
    outputs = model(X_train_t)
    loss = criterion(outputs, y_train_t)
    loss.backward()
    optimizer.step()
    train_losses.append(loss.item())
    if (epoch+1) % 20 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

Epoch [20/200], Loss: 0.2295
Epoch [40/200], Loss: 0.1438
Epoch [60/200], Loss: 0.1175
Epoch [80/200], Loss: 0.1041
Epoch [100/200], Loss: 0.0953
Epoch [120/200], Loss: 0.0888
Epoch [140/200], Loss: 0.0838
Epoch [160/200], Loss: 0.0798
Epoch [180/200], Loss: 0.0764
Epoch [200/200], Loss: 0.0736


In [10]:
model.eval()
with torch.no_grad():
    y_pred_prob = model(X_test_t)
    y_pred = (y_pred_prob >= 0.5).float()

accuracy = (y_pred == y_test_t).float().mean()
print(f'Accuracy: {accuracy:.4f}')

Accuracy: 0.9649


In [11]:
from tool import plot_loss_and_confusion
plot_loss_and_confusion(train_losses, 
                        y_test_t.cpu(), 
                        y_pred.cpu(), 
                        data.target_names, 
                        ylabel='BCE Loss')

ModuleNotFoundError: No module named 'tool'